
# Assignment 3.1 — Cleaning and Reshaping UN Immigration Data

**Course:** Doctorate in Computer Science  
**Topic:** Data preparation with Python and Pandas

## Purpose

The goal of this assignment is to take the UN immigration dataset supplied in the assignment, remove metadata that is not needed for the requested analysis, rename the descriptive columns, standardize the year columns, and calculate a **Total** column containing the total number of immigrants for each country from **1980 through 2013**.

The final DataFrame is organized as requested in the assignment:

`Country | Continent | Region | DevName | 1980 ... 2013 | Total`



## 1. Import the required libraries

Pandas is used for reading and transforming the tabular data. No visualization library is required because this assignment focuses on data preparation.


In [1]:

import pandas as pd

print("Pandas version:", pd.__version__)


Pandas version: 2.2.3



## 2. Read the immigration data

The instructor provided the immigration dataset through a Google Sheets link. The `gviz` CSV endpoint allows Pandas to read the first worksheet directly.

The code below keeps the source location in one variable so that it is easy to update if the instructor changes the spreadsheet.


In [2]:

# Google Sheet supplied in Assignment 3.1
sheet_id = "10Gs8KoHGEDZxyFdaxAKmZbeO1BiSWVQ6"
source_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv"

# Read the source worksheet into a DataFrame.
df_raw = pd.read_csv(source_url)

print("Raw DataFrame shape:", df_raw.shape)
df_raw.head()


Raw DataFrame shape: (195, 43)



              Type   Coverage          OdName  AREA  AreaName   REG          RegName  DEV            DevName  1980  1981  ...  2004  2005  2006  2007  2008  2009  2010  2011  2012  2013
0       Immigrants  Foreigners     Afghanistan   935      Asia  5501    Southern Asia  902  Developing regions    16    39  ...  2978  3436  3009  2652  2111  1746  1758  2203  2635  2004
1       Immigrants  Foreigners         Albania   908    Europe   925  Southern Europe  901   Developed regions     1     0  ...  1450  1223   856   702   560   716   561   539   620   603
2       Immigrants  Foreigners         Algeria   903    Africa   912   Northern Africa  902  Developing regions    80    67  ...  3616  3626  4807  3623  4005  5393  4752  4325  3774  4331
3       Immigrants  Foreigners  American Samoa   909   Oceania   957       Polynesia  902  Developing regions     0     1  ...     0     0     1     0     0     0     0     0     0     0
4       Immigrants  Foreigners         Andorra   908    Europ


## 3. Remove unnecessary metadata columns

The requested final structure does not need the following source columns:

- `Type`
- `Coverage`
- `AREA`
- `REG`
- `DEV`

The useful descriptive fields are retained so that each country still has its continent, region, and development classification.


In [3]:

# Remove source metadata that is not part of the requested final table.
columns_to_drop = ["Type", "Coverage", "AREA", "REG", "DEV"]
df = df_raw.drop(columns=columns_to_drop).copy()

print("Shape after removing unnecessary metadata:", df.shape)


Shape after removing unnecessary metadata: (195, 38)



## 4. Rename the descriptive columns

The original UN dataset uses names such as `OdName`, `AreaName`, and `RegName`. The assignment's target format uses the clearer names `Country`, `Continent`, and `Region`.


In [4]:

# Rename source fields to match the requested target format.
df = df.rename(columns={
    "OdName": "Country",
    "AreaName": "Continent",
    "RegName": "Region"
})

print(df.columns[:4].tolist())


['Country', 'Continent', 'Region', 'DevName']



## 5. Standardize the year columns and calculate Total

The immigration counts cover the years **1980–2013**. These 34 columns should contain numeric values. Converting them explicitly to numeric values makes the calculation reliable.

The `Total` column is calculated by summing only the yearly immigration counts. This avoids accidentally adding non-numeric descriptive fields.


In [5]:

# Create the exact list of year columns required by the assignment.
years = [str(year) for year in range(1980, 2014)]

# Convert every yearly immigration column to numeric values.
# Any non-numeric value is treated as missing and replaced with 0.
for year in years:
    df[year] = pd.to_numeric(df[year], errors="coerce").fillna(0).astype(int)

# Calculate total immigration for each country over 1980–2013.
df["Total"] = df[years].sum(axis=1)

print("Number of year columns:", len(years))
print("First year:", years[0])
print("Last year:", years[-1])


Number of year columns: 34
First year: 1980
Last year: 2013



## 6. Arrange the columns in the requested order

The final table should begin with the four descriptive columns, followed by the yearly immigration counts, and end with `Total`.


In [6]:

# Arrange the final DataFrame exactly as requested.
final_columns = ["Country", "Continent", "Region", "DevName"] + years + ["Total"]
df = df[final_columns]

print("Final DataFrame shape:", df.shape)
df.head()


Final DataFrame shape: (195, 39)



          Country Continent          Region            DevName  1980  1981  1982  1983  1984  1985  1986  ...  2005  2006  2007  2008  2009  2010  2011  2012  2013  Total
0     Afghanistan      Asia   Southern Asia  Developing regions    16    39    39    47    71   340   496  ...  3436  3009  2652  2111  1746  1758  2203  2635  2004  58639
1         Albania    Europe  Southern Europe  Developed regions     1     0     0     0     0     0     1  ...  1223   856   702   560   716   561   539   620   603  15699
2         Algeria    Africa   Northern Africa  Developing regions    80    67    71    69    63    44    69  ...  3626  4807  3623  4005  5393  4752  4325  3774  4331  69439
3  American Samoa   Oceania       Polynesia  Developing regions     0     1     0     0     0     0     0  ...     0     1     0     0     0     0     0     0     0      6
4         Andorra    Europe  Southern Europe  Developed regions     0     0     0     0     0     0     2  ...     0     1     1     0    


## 7. Validate the result

The following checks confirm that the transformation produced the expected structure:

1. There are 195 country records.
2. There are 39 final columns: 4 descriptive columns + 34 years + `Total`.
3. All requested column names are present.
4. The `Total` value for each country equals the sum of its yearly values.


In [7]:

expected_first_columns = ["Country", "Continent", "Region", "DevName"]
expected_columns = expected_first_columns + years + ["Total"]

shape_check = df.shape == (195, 39)
column_check = df.columns.tolist() == expected_columns
country_count_check = df["Country"].nunique() == 195

# Recalculate totals independently and compare them with the Total column.
total_check = df["Total"].equals(df[years].sum(axis=1))

print("Shape check:", shape_check)
print("Column-order check:", column_check)
print("Country-count check:", country_count_check)
print("Total calculation check:", total_check)


Shape check: True
Column-order check: True
Country-count check: True
Total calculation check: True



## 8. Final result

The data has been successfully cleaned and converted into the requested structure. The final DataFrame contains country, continent, region, development classification, annual immigration counts from 1980 through 2013, and a calculated total for the full period.

The first five total values provide a quick verification of the transformation:

| Country | Total immigration (1980–2013) |
|---|---:|
| Afghanistan | 58,639 |
| Albania | 15,699 |
| Algeria | 69,439 |
| American Samoa | 6 |
| Andorra | 15 |

### Conclusion

This transformation makes the UN immigration dataset easier to analyze because unnecessary identifier fields have been removed, descriptive fields have been renamed for clarity, yearly values have been standardized as integers, and a total immigration measure has been added for each country. The final structure matches the target format shown in the assignment instructions.
